In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf
from IPython.display import display, HTML

clean = Path("data/clean")
results = Path("results")
results.mkdir(exist_ok=True)

# -----------------------------
# Load combined price + population data
# -----------------------------

wide = pd.read_csv(clean / "population_density_all_suburbs.csv")

wide["suburb"] = wide["suburb"].astype(str).str.strip().str.upper()
wide["treated"] = wide["treated"].astype(int)

# -----------------------------
# Reshape price and population columns to long format
# -----------------------------

price_cols = [col for col in wide.columns if col.endswith("_price")]
pop_cols = [col for col in wide.columns if col.endswith("_pop")]

prices_long = wide[["suburb", "status", "treated"] + price_cols].melt(
    id_vars=["suburb", "status", "treated"],
    value_vars=price_cols,
    var_name="year",
    value_name="price"
)

prices_long["year"] = prices_long["year"].str.replace("_price", "", regex=False).astype(int)

population_long = wide[["suburb"] + pop_cols].melt(
    id_vars=["suburb"],
    value_vars=pop_cols,
    var_name="year",
    value_name="population"
)

population_long["year"] = population_long["year"].str.replace("_pop", "", regex=False).astype(int)

# -----------------------------
# Merge final panel
# -----------------------------

df = (
    prices_long
    .merge(population_long, on=["suburb", "year"], how="inner")
    .dropna(subset=["price", "population", "treated"])
    .copy()
)

df = df[(df["price"] > 0) & (df["population"] > 0)].copy()

# -----------------------------
# DiD variables
# -----------------------------

df["post"] = (df["year"] >= 2022).astype(int)
df["treat_post"] = df["treated"] * df["post"]

df["log_price"] = np.log(df["price"])
df["log_population"] = np.log(df["population"])

# -----------------------------
# Run DiD regression
# -----------------------------

model = smf.ols(
    "log_price ~ treat_post + log_population + C(suburb) + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["suburb"]}
)

# -----------------------------
# Build table
# -----------------------------

def get_row(label, variable):
    if variable not in model.params.index:
        return [label, "Absorbed", "", "", "", ""]

    coef = model.params[variable]
    se = model.bse[variable]
    t_val = model.tvalues[variable]
    p_val = model.pvalues[variable]
    ci_low, ci_high = model.conf_int().loc[variable]

    return [
        label,
        f"{coef:.3f}",
        f"{se:.3f}",
        f"{t_val:.3f}",
        f"{p_val:.3f}",
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    ]

did_table = pd.DataFrame(
    [
        get_row("Intercept", "Intercept"),
        ["Treat", "Absorbed by suburb FE", "", "", "", ""],
        ["Post", "Absorbed by year FE", "", "", "", ""],
        get_row("Treat x Post (DiD effect)", "treat_post"),
        get_row("log_population", "log_population"),
        ["Suburb FE", "Yes", "", "", "", ""],
        ["Year FE", "Yes", "", "", "", ""],
        ["N", f"{int(model.nobs):,}", "", "", "", ""],
        ["R²", f"{model.rsquared:.3f}", "", "", "", ""],
    ],
    columns=["Variable", "Coef", "Std Err", "t", "p-value", "CI"]
)

# -----------------------------
# Display formatted table
# -----------------------------

html = did_table.to_html(index=False, escape=False)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:separate; border-spacing:0; width:92%; font-family:Arial, sans-serif; font-size:16px; color:#2b2927; border:1px solid #dcc7b3; border-radius:10px; overflow:hidden;">'
)

html = html.replace(
    "<thead>",
    '<thead style="background-color:#f7eee5;">'
)

html = html.replace(
    "<th>",
    '<th style="padding:14px 18px; text-align:left; font-weight:700; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

html = html.replace(
    "<td>",
    '<td style="padding:14px 18px; text-align:left; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

display(HTML(html))

# -----------------------------
# Save table
# -----------------------------

did_table.to_csv(results / "new_did_regression_table.csv", index=False)

with open(results / "new_did_regression_table.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved:")
print(results / "new_did_regression_table.csv")
print(results / "new_did_regression_table.html")

# -----------------------------
# Interpretation values
# -----------------------------

did_coef = model.params["treat_post"]
did_se = model.bse["treat_post"]
did_p = model.pvalues["treat_post"]
did_pct = (np.exp(did_coef) - 1) * 100

print()
print("Sample used:")
print("Suburbs:", df["suburb"].nunique())
print("Observations:", len(df))
print("Treated suburbs:", df.loc[df["treated"] == 1, "suburb"].nunique())
print("Control suburbs:", df.loc[df["treated"] == 0, "suburb"].nunique())

print()
print("Main DiD result:")
print(f"Treat x Post coefficient: {did_coef:.3f}")
print(f"Clustered standard error: {did_se:.3f}")
print(f"p-value: {did_p:.3f}")
print(f"Approximate percentage effect: {did_pct:.1f}%")


Variable,Coef,Std Err,t,p-value,CI
Intercept,10.662,0.655,16.282,0.000,"[9.378, 11.945]"
Treat,Absorbed by suburb FE,,,,
Post,Absorbed by year FE,,,,
Treat x Post (DiD effect),-0.075,0.030,-2.484,0.013,"[-0.134, -0.016]"
log_population,0.300,0.069,4.327,0.000,"[0.164, 0.436]"
Suburb FE,Yes,,,,
Year FE,Yes,,,,
N,490,,,,
R²,0.983,,,,


Saved:
results\new_did_regression_table.csv
results\new_did_regression_table.html

Sample used:
Suburbs: 49
Observations: 490
Treated suburbs: 24
Control suburbs: 25

Main DiD result:
Treat x Post coefficient: -0.075
Clustered standard error: 0.030
p-value: 0.013
Approximate percentage effect: -7.2%


Interpretation

The main coefficient is Treat x Post, which is the DiD estimate. In this model, the coefficient is approximately -0.075.

Because the dependent variable is log_price, this is interpreted in percentage terms. Converting from log points:

exp(-0.075) - 1 = -0.072
So the estimated effect is about -7.2%.

This means that after 2022, treated suburbs experienced about a 7.2% lower relative change in median house prices compared with untreated suburbs.

The direction is negative, meaning treated suburbs had weaker price growth relative to the control group after the post period began. The magnitude is 7.2%, and the units are percentage change in median house prices.

This estimate holds constant:

- log_population, so the comparison accounts for population differences over time.

- Suburb fixed effects, so it controls for fixed suburb characteristics such as location, baseline house prices, and long-run desirability.

- Year fixed effects, so it controls for common market shocks affecting all suburbs in each year, such as interest rates, inflation, and broad housing-market trends.

The p-value is about 0.013, so the DiD estimate is statistically significant at the 5% level.